# Семинар 1. Чтение данных из S3

Курсовой проект ML/DLOps · Inside Airbnb

В этом ноутбуке нужно заполнить **две функции**. Импорты, клиент S3 и примеры вызова уже готовы.

1. Сохраните `01_read_data.ipynb` в корне своего репозитория.
2. Подставьте bucket и key, которые выдаст преподаватель и положите копию того же CSV в `data/train.csv`.
3. Выполняйте ячейки сверху вниз. Сначала заполните TODO 1, затем TODO 2.

Прежде чем реализовывать функцию, удалите `NotImplementedError`.

## Окружение и запуск

Команды выполняются **в терминале**, из папки репозитория:

```bash
pip install boto3
```



## Импорты и настройки

Подставьте bucket и key, которые выдаст преподаватель.
Импортируйте KEY_ID и SECRET_KEY из файла.
Не забудьте добавить файл в .gitignore

In [1]:
from pathlib import Path

import boto3
import pandas as pd
from botocore.exceptions import BotoCoreError, ClientError

from secretkeys import ACCESS_KEY, KEY_ID

# Замените примеры значениями преподавателя.
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/train.csv"
LOCAL_PATH = Path("data/train.csv")

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

## TODO 1. Чтение CSV из S3

Заполните `read_s3_csv(bucket, key)`. Функция принимает имя bucket и полный ключ объекта и возвращает `DataFrame`.

Например, для `s3://course-bucket/prepared/listings.csv`:
- bucket: `course-bucket`;
- key: `prepared/listings.csv`.

После изменения определения функции выполните эту ячейку заново.

Ответ от s3 будет в `response["Body"]`.

In [2]:
def read_s3_csv(bucket, key):
    """Прочитайте CSV из S3 и верните DataFrame.

    1. Получите объект: s3.get_object(Bucket=bucket, Key=key).
    2. Прочитайте CSV через pd.read_csv(body).
    """
    # Напишите реализацию вместо следующей строки.
    response = s3.get_object(Bucket=bucket, Key=key)
    df = pd.read_csv(response["Body"])
    return df

### Первое чтение

Выполняйте после TODO 1. Проверьте первые строки, размер и названия колонок.
Если возникла ошибка S3, проверьте адрес и доступ к объекту.

In [12]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)
print("Колонки:", df_s3.columns.tolist())

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,1001248165793428398,2026-03-21,51.434110,-0.161020,2,1.0,1.0,1.0,1.0,Wandsworth,Entire serviced apartment,Entire home/apt,130.0
1,1001501494788436614,2026-03-21,51.514120,-0.176810,5,1.0,3.0,3.0,2.0,Westminster,Entire rental unit,Entire home/apt,232.0
2,1001555419418023705,2026-03-21,51.542042,0.121361,4,2.0,2.0,2.0,1.0,Barking and Dagenham,Entire condo,Entire home/apt,216.0
3,1001555573812356101,2026-03-21,51.540557,0.120994,2,1.0,1.0,1.0,1.0,Barking and Dagenham,Entire place,Entire home/apt,115.0
4,1001604905732387546,2026-03-21,51.377860,-0.086780,2,1.0,1.0,1.0,1.0,Croydon,Private room in home,Private room,86.0


Размер таблицы: (5724, 13)
Колонки: ['listing_id', 'snapshot_date', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'neighbourhood_cleansed', 'property_type', 'room_type', 'price']


In [20]:
df_s3.to_csv('train.csv', index=False)

## TODO 2. Обработка ошибок и локальная копия

Заполните `read_data`. При доступном S3 возвращаем удалённые данные.
При сбое доступа или сети читаем локальный CSV, если путь передан.
Сообщение о переходе к файлу должно быть видно в выводе ячейки.

Ошибки формата CSV должны оставаться видимыми.

In [13]:
def read_data(bucket, key, local_path=None):
    """Попробуйте S3, при сбое доступа или сети прочитайте локальную копию.

    1. В try вызовите read_s3_csv(bucket, key) и верните результат.
    2. Перехватите BotoCoreError, ClientError и OSError.
    3. Если local_path не передан, повторно поднимите исходную ошибку: raise.
    4. Иначе выведите тип ошибки и сообщение о переходе к локальному файлу.
    5. Прочитайте local_path через pd.read_csv.
       Верните DataFrame.

    Не используйте except Exception. Ошибки разбора CSV должны быть видны.
    Если локального файла нет, должен возникнуть FileNotFoundError.
    """
    # Напишите реализацию вместо следующей строки.
    try:
        return read_s3_csv(bucket, key)
    except (BotoCoreError, ClientError, OSError) as error:
        if local_path is None:
            raise
        print(f"{type(error).__name__}: {error}")
        print(f"Не удалось прочитать из S3, читаем локальный файл {local_path}")
        return pd.read_csv(local_path)

### Проверка: S3 доступен

Ожидаем таблицу из S3 без сообщения о переходе к локальному файлу.

In [14]:
df = read_data(S3_BUCKET, S3_KEY, LOCAL_PATH)
display(df.head())
print("Размер:", df.shape)

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,1001248165793428398,2026-03-21,51.434110,-0.161020,2,1.0,1.0,1.0,1.0,Wandsworth,Entire serviced apartment,Entire home/apt,130.0
1,1001501494788436614,2026-03-21,51.514120,-0.176810,5,1.0,3.0,3.0,2.0,Westminster,Entire rental unit,Entire home/apt,232.0
2,1001555419418023705,2026-03-21,51.542042,0.121361,4,2.0,2.0,2.0,1.0,Barking and Dagenham,Entire condo,Entire home/apt,216.0
3,1001555573812356101,2026-03-21,51.540557,0.120994,2,1.0,1.0,1.0,1.0,Barking and Dagenham,Entire place,Entire home/apt,115.0
4,1001604905732387546,2026-03-21,51.377860,-0.086780,2,1.0,1.0,1.0,1.0,Croydon,Private room in home,Private room,86.0


Размер: (5724, 13)


### Проверка: объект не найден

Намеренно меняем ключ. Ожидаем сообщение о переходе к `data/train.csv` и таблицу из локальной копии.
Перед запуском убедитесь, что локальный файл существует.

In [21]:
df_local = read_data(S3_BUCKET, S3_KEY + ".missing", LOCAL_PATH)
display(df_local.head())
print("Размер:", df_local.shape)

NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.
Не удалось прочитать из S3, читаем локальный файл data/train.csv


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,1001248165793428398,2026-03-21,51.434110,-0.161020,2,1.0,1.0,1.0,1.0,Wandsworth,Entire serviced apartment,Entire home/apt,130.0
1,1001501494788436614,2026-03-21,51.514120,-0.176810,5,1.0,3.0,3.0,2.0,Westminster,Entire rental unit,Entire home/apt,232.0
2,1001555419418023705,2026-03-21,51.542042,0.121361,4,2.0,2.0,2.0,1.0,Barking and Dagenham,Entire condo,Entire home/apt,216.0
3,1001555573812356101,2026-03-21,51.540557,0.120994,2,1.0,1.0,1.0,1.0,Barking and Dagenham,Entire place,Entire home/apt,115.0
4,1001604905732387546,2026-03-21,51.377860,-0.086780,2,1.0,1.0,1.0,1.0,Croydon,Private room in home,Private room,86.0


Размер: (5724, 13)


### Дополнительная ручная проверка

Раскомментируйте **один** вызов за раз:

- нет объекта и не передан запасной путь — исходная ошибка S3;
- нет объекта и локального файла — `FileNotFoundError`.

Эти вызовы должны завершаться ошибкой. После проверки закомментируйте их обратно.

In [ ]:
# read_data(S3_BUCKET, S3_KEY + ".missing")
# read_data(S3_BUCKET, S3_KEY + ".missing", "data/absent.csv")

## Перед pull request

- Обе функции заполнены, обычное чтение и переход к файлу проверены.
- В README указаны pip-команды, источник данных и ваша обычная среда открытия ноутбука.
- В `.gitignore` добавлены `data/` и `.ipynb_checkpoints/`.
- В коммит входят `01_read_data.ipynb`, README и `.gitignore`. CSV и ключи доступа остаются локально.
- В `.gitignore` есть файл secretkeys.py

На следующих занятиях продолжаем этот же курсовой проект.